# Loading Dataset

In [1]:
import pandas as pd

# Menampilkan ringkasan informasi dari dataset
df = pd.read_csv('noice_app_reviews.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12223 entries, 0 to 12222
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   reviewId              12223 non-null  str  
 1   userName              12223 non-null  str  
 2   userImage             12223 non-null  str  
 3   content               12223 non-null  str  
 4   score                 12223 non-null  int64
 5   thumbsUpCount         12223 non-null  int64
 6   reviewCreatedVersion  10275 non-null  str  
 7   at                    12223 non-null  str  
 8   replyContent          3594 non-null   str  
 9   repliedAt             3594 non-null   str  
 10  appVersion            10275 non-null  str  
dtypes: int64(2), str(9)
memory usage: 1.0 MB


In [2]:
# Membuat salinan dari DataFrame dengan hanya kolom 'content' dan 'score'
df_clean = df[['content', 'score']].copy()

# Menghapus baris yang memiliki nilai NaN pada kolom 'content' atau 'score'
df_clean = df_clean.dropna(subset=['content', 'score'])

# Menghapus duplikat dari DataFrame dan mereset indeks
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

print(f"Total data bersih: {df_clean.shape}\n")
print(df_clean.info())

Total data bersih: (10485, 2)

<class 'pandas.DataFrame'>
RangeIndex: 10485 entries, 0 to 10484
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   content  10485 non-null  str  
 1   score    10485 non-null  int64
dtypes: int64(1), str(1)
memory usage: 164.0 KB
None


# Teks Preprocessing

In [3]:
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\IRUL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\IRUL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\IRUL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
 
    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text
 
# Mengubah semua karakter dalam teks menjadi huruf kecil
def casefoldingText(text):
    text = text.lower()
    return text
 
# Memecah atau membagi string, teks menjadi daftar token
def tokenizingText(text):
    text = word_tokenize(text)
    return text

# Menghapus stopwords dalam teks
def filteringText(text): 
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text
 
# Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
def stemmingText(text):
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
 
    # Memecah teks menjadi daftar kata
    words = text.split()
 
    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]
 
    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)
 
    return stemmed_text

# Mengubah daftar kata menjadi kalimat
def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence

In [5]:
slangwords = {"@": "di", "apk": "aplikasi", "gua": "saya", "sy": "saya", "td": "tadi", "dikit": "sedikit", "ud": "sudah", "udh": "sudah", "sdh": "sudah", "abis": "habis", "aja": "saja", "masi": "masih", "bgt": "banget", "gak": "tidak", "gk": "gak", "ga": "tidak", 
              "pake": "pakai", "jgn": "jangan", "yg": "yang", "klo": "kalau", "kalo": "kalau", "gmn": "bagaimana", "gitu": "begitu", "dgn": "dengan", "dr": "dari", "sm": "sama", "sbg": "sebagai", "sblm": "sebelum", "knp": "kenapa", "hp": "telepon", "balikin": "kembalikan", 
              "ngga": "tidak", "nggak": "tidak", "nih": "ini", "kmn": "kemana", "mulu": "selalu", "bgtt": "banget", "gw": "saya", "boong": "bohong", "kh": "ya", "gapapa": "tidak apa-apa", "gpp": "tidak apa-apa", "please": "tolong", "plis": "tolong", "pliss": "tolong", 
              "bug": "kesalahan", "bgtu": "begitu", "trs": "terus", "trus": "terus", "play": "dimainkan", "ngulang": "mengulang", "nntn": "menonton", "nonton": "menonton", "dengerin": "mendengarkan", "udah": "sudah", "buat": "untuk", "cuma": "hanya", "pas": "saat", "denger": "mendengar", 
              "karna": "karena", "makin": "semakin", "gabisa": "tidak bisa", "bikin": "membuat", "biar": "agar", "dah": "sudah", "tau": "tahu", "gimana": "bagaimana", "tp": "tapi", "doang": "saja", "pengen": "ingin", "cuman": "hanya", "tetep": "tetap", "gue": "saya", "ni": "ini", 
              "sampe": "sampai", "lemot": "lambat", "muter": "memutar", "kaya": "seperti", "tambahin": "tambahkan", "bener": "benar", "nyesel": "menyesal", "yaa": "ya", "mantab": "mantap", "emang": "memang", "lg": "lagi", "kayak": "seperti", "jd": "jadi", "tpi": "tapi", "gini": "begini",
              "temen": "teman", "mending": "lebih baik", "liat": "melihat", "males": "malas", "mantep": "mantap", "mantul": "mantap", "mantapp": "mantap", "baguss": "bagus", "dapet": "dapat", "jg": "juga", "dlu": "dulu", "makasih": "terima kasih", "krn": "karena", "nemenin": "menemani", 
              "nomer": "nomor", "gokil": "keren", "pdhl": "padahal","ko": "kok", "gabut": "bosan", "lgi": "lagi", "benerin": "perbaiki", "ngelag": "tersendat", "ilang": "hilang", "seneng": "senang", "wkwk": "tertawa", "hehe": "tertawa", "gaada": "tidak ada", "ny": "nya", "nunggu": "menunggu", 
              "ngebug": "kesalahan", "kek": "seperti", "jdi": "jadi", "nyoba": "mencoba", "yah": "ya"
              }
def fix_slangwords(text):
    words = text.split()
    fixed_words = []
 
    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)
 
    fixed_text = ' '.join(fixed_words)
    return fixed_text

In [6]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
df_clean['text_clean'] = df_clean['content'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
df_clean['text_casefoldingText'] = df_clean['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
df_clean['text_slangwords'] = df_clean['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
df_clean['text_tokenizingText'] = df_clean['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
df_clean['text_stopword'] = df_clean['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_final'
df_clean['text_final'] = df_clean['text_stopword'].apply(toSentence)

# Labelling Dataset

In [55]:
# Pelabelan sentimen berdasarkan skor ulasan
def sentiment_label(score):
    if score >= 4:
        return 'positive'
    else:
        return 'negative'

# Membuat kolom baru 'sentiment' berdasarkan skor ulasan
df_clean['sentiment'] = df_clean['score'].apply(sentiment_label)

# Menampilkan distribusi sentiment dalam dataset
print(df_clean['sentiment'].value_counts())

sentiment
positive    7081
negative    3404
Name: count, dtype: int64


In [56]:
df_clean.head()

,content,score,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_final,sentiment
0,"aplikasinya bagus, ada podcast favorit saya, d...",5,aplikasinya bagus ada podcast favorit saya dan...,aplikasinya bagus ada podcast favorit saya dan...,aplikasinya bagus ada podcast favorit saya dan...,"[aplikasinya, bagus, ada, podcast, favorit, sa...","[aplikasinya, bagus, podcast, favorit, fitur, ...",aplikasinya bagus podcast favorit fitur downlo...,positive
1,"jujur sebenarnya bagus kok aplikasinya, kita m...",1,jujur sebenarnya bagus kok aplikasinya kita me...,jujur sebenarnya bagus kok aplikasinya kita me...,jujur sebenarnya bagus kok aplikasinya kita me...,"[jujur, sebenarnya, bagus, kok, aplikasinya, k...","[jujur, bagus, aplikasinya, mendengarkan, podc...",jujur bagus aplikasinya mendengarkan podcast t...,negative
2,kecewa sm aplikasinya sudah download tp gak bi...,1,kecewa sm aplikasinya sudah download tp gak bi...,kecewa sm aplikasinya sudah download tp gak bi...,kecewa sama aplikasinya sudah download tapi ti...,"[kecewa, sama, aplikasinya, sudah, download, t...","[kecewa, aplikasinya, download, buka, berlangg...",kecewa aplikasinya download buka berlangganan ...,negative
3,Aku naikkan bintangku karena konten Deddy Issu...,4,Aku naikkan bintangku karena konten Deddy Issu...,aku naikkan bintangku karena konten deddy issu...,aku naikkan bintangku karena konten deddy issu...,"[aku, naikkan, bintangku, karena, konten, dedd...","[naikkan, bintangku, konten, deddy, issues, te...",naikkan bintangku konten deddy issues terjangk...,positive
4,sy download video tapi ternyata yg terdownload...,1,sy download video tapi ternyata yg terdownload...,sy download video tapi ternyata yg terdownload...,saya download video tapi ternyata yang terdown...,"[saya, download, video, tapi, ternyata, yang, ...","[download, video, terdownload, audio, tolong, ...",download video terdownload audio tolong tombol...,negative


# Feature Extraction

In [94]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import accuracy_score

In [95]:
# Memisahkan data menjadi Fitur (X) dan Target (y)
X = df_clean['text_final']
y = df_clean['sentiment']

# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=5000, min_df=3, max_df=0.8, ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(X)

# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
 
# Menampilkan hasil ekstraksi fitur
features_df

,aamiin,abang,abdur,acak,acara,acaranya,ad,adain,adain fitur,adakah,...,youtube ig,youtube lancar,youtube spotify,yt,yt lancar,yu,yuk,yuk download,yutub,zaman
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10481,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10482,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [96]:
# Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Membuat objek model Naive Bayes (Multinomial Naive Bayes)
naive_bayes = MultinomialNB()

# Melatih model Naive Bayes pada data pelatihan
naive_bayes.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = naive_bayes.predict(X_train.toarray())
y_pred_test_nb = naive_bayes.predict(X_test.toarray())

# Evaluasi akurasi model Naive Bayes
accuracy_train_nb = accuracy_score(y_pred_train_nb, y_train)
accuracy_test_nb = accuracy_score(y_pred_test_nb, y_test)

# Menampilkan akurasi
print('Naive Bayes - accuracy_train:', accuracy_train_nb)
print('Naive Bayes - accuracy_test:', accuracy_test_nb)

Naive Bayes - accuracy_train: 0.8848354792560801
Naive Bayes - accuracy_test: 0.8578922269909395


# Random Forest

In [98]:
from sklearn.ensemble import RandomForestClassifier

# Membuat objek model Random Forest
random_forest = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())

# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9761564139246542
Random Forest - accuracy_test: 0.8478779208392943


# Logistic Regression

In [99]:
from sklearn.linear_model import LogisticRegression

# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression(class_weight='balanced', max_iter=1000)

# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train.toarray())
y_pred_test_lr = logistic_regression.predict(X_test.toarray())

# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)

# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)

# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Logistic Regression - accuracy_train: 0.8913924654268002
Logistic Regression - accuracy_test: 0.8617072007629948


# Decision Tree

In [100]:
from sklearn.tree import DecisionTreeClassifier

# Membuat objek model Decision Tree
decision_tree = DecisionTreeClassifier()

# Melatih model Decision Tree pada data pelatihan
decision_tree.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_dt = decision_tree.predict(X_train.toarray())
y_pred_test_dt = decision_tree.predict(X_test.toarray())

# Evaluasi akurasi model Decision Tree
accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test)

# Menampilkan akurasi
print('Decision Tree - accuracy_train:', accuracy_train_dt)
print('Decision Tree - accuracy_test:', accuracy_test_dt)

Decision Tree - accuracy_train: 0.9761564139246542
Decision Tree - accuracy_test: 0.8187887458273725


In [ ]:
# Membuat DataFrame untuk hasil akurasi
results_df = pd.DataFrame({
    'Model': ['Naive Bayes', 'Random Forest', 'Logistic Regression', 'Decision Tree'],
    'Accuracy Train': [accuracy_train_nb, accuracy_train_rf, accuracy_train_lr, accuracy_train_dt],
    'Accuracy Test': [accuracy_test_nb, accuracy_test_rf, accuracy_test_lr, accuracy_test_dt]
})

# Menampilkan hanya kolom "Accuracy Test"
accuracy_test_only = results_df[['Model', 'Accuracy Test']]
print(accuracy_test_only)

                 Model  Accuracy Test
0          Naive Bayes       0.857892
1        Random Forest       0.847878
2  Logistic Regression       0.861707
3        Decision Tree       0.818789


In [103]:
# Mengurutkan DataFrame berdasarkan kolom "Accuracy Test" dari tertinggi ke terendah
accuracy_test_sorted = accuracy_test_only.sort_values(by='Accuracy Test', ascending=False)

# Menampilkan DataFrame yang telah diurutkan
print(accuracy_test_sorted)

                 Model  Accuracy Test
2  Logistic Regression       0.861707
0          Naive Bayes       0.857892
1        Random Forest       0.847878
3        Decision Tree       0.818789
